In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

In [2]:
load_dotenv()

openai_endpoint = os.getenv("ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

In [4]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model=model_deployment,
    base_url=openai_endpoint,
    api_key=token_provider()
)

In [5]:
import asyncio
from autogen_agentchat.agents import AssistantAgent


# ---------------------------------------------------------
# 1. TASK
# ---------------------------------------------------------

task = """
Write a concise but engaging blog post about DeepLearning.AI.
Include a title and keep the complete blog post within 100 words.
"""


# ---------------------------------------------------------
# 2. CREATE THE WRITER
# ---------------------------------------------------------

writer = AssistantAgent(
    name="Writer",
    model_client=model_client,
    system_message=(
        "You are a professional writer. "
        "Write engaging and concise blog posts with a title. "
        "When you receive reviewer feedback, polish your writing and "
        "provide a refined version. "
        "Only return the blog post without additional comments."
    ),
)


# ---------------------------------------------------------
# 3. CREATE THE MAIN CRITIC
# ---------------------------------------------------------

critic = AssistantAgent(
    name="Critic",
    model_client=model_client,
    system_message=(
        "You are a content critic. Review the writer's work and provide "
        "constructive, specific, and concise feedback that will improve "
        "clarity, engagement, structure, and writing quality."
    ),
)


# ---------------------------------------------------------
# 4. CREATE SPECIALIST REVIEWERS
# ---------------------------------------------------------

seo_reviewer = AssistantAgent(
    name="SEO_Reviewer",
    model_client=model_client,
    system_message=(
        "You are an SEO reviewer. Review content for search visibility, "
        "clear keywords, title quality, and readability. "
        "Return no more than 3 concise bullet points. "
        "Begin your response with 'SEO Reviewer:'."
    ),
)


legal_reviewer = AssistantAgent(
    name="Legal_Reviewer",
    model_client=model_client,
    system_message=(
        "You are a legal reviewer. Identify unsupported claims, possible "
        "misrepresentation, attribution concerns, or other legal risks. "
        "Return no more than 3 concise bullet points. "
        "Begin your response with 'Legal Reviewer:'."
    ),
)


ethics_reviewer = AssistantAgent(
    name="Ethics_Reviewer",
    model_client=model_client,
    system_message=(
        "You are an ethics reviewer. Check whether the content is fair, "
        "responsible, inclusive, transparent, and free from misleading claims. "
        "Return no more than 3 concise bullet points. "
        "Begin your response with 'Ethics Reviewer:'."
    ),
)


meta_reviewer = AssistantAgent(
    name="Meta_Reviewer",
    model_client=model_client,
    system_message=(
        "You are a meta-reviewer. Aggregate feedback from the critic, "
        "SEO reviewer, legal reviewer, and ethics reviewer. "
        "Remove duplicate or conflicting recommendations and provide "
        "a concise final set of actionable suggestions for the writer."
    ),
)


# ---------------------------------------------------------
# 5. GENERATE THE FIRST DRAFT
# ---------------------------------------------------------

draft_result = await writer.run(task=task)
draft = draft_result.messages[-1].content

print("=" * 70)
print("FIRST DRAFT")
print("=" * 70)
print(draft)


# ---------------------------------------------------------
# 6. RUN THE CRITIC
# ---------------------------------------------------------

critic_result = await critic.run(
    task=f"""
Review the following blog post.

Original requirements:
{task}

Blog post:
{draft}
"""
)

critic_feedback = critic_result.messages[-1].content


# ---------------------------------------------------------
# 7. RUN SPECIALIST REVIEWS IN PARALLEL
# ---------------------------------------------------------

review_prompt = f"""
Review the following blog post.

Original requirements:
{task}

Blog post:
{draft}
"""

seo_result, legal_result, ethics_result = await asyncio.gather(
    seo_reviewer.run(task=review_prompt),
    legal_reviewer.run(task=review_prompt),
    ethics_reviewer.run(task=review_prompt),
)

seo_feedback = seo_result.messages[-1].content
legal_feedback = legal_result.messages[-1].content
ethics_feedback = ethics_result.messages[-1].content


# ---------------------------------------------------------
# 8. AGGREGATE ALL REVIEWS
# ---------------------------------------------------------

meta_result = await meta_reviewer.run(
    task=f"""
Aggregate the following feedback and provide one concise set of final
recommendations for the writer.

Original blog post:
{draft}

General critic feedback:
{critic_feedback}

SEO feedback:
{seo_feedback}

Legal feedback:
{legal_feedback}

Ethics feedback:
{ethics_feedback}
"""
)

final_feedback = meta_result.messages[-1].content


# ---------------------------------------------------------
# 9. ASK THE WRITER TO REFINE THE BLOG POST
# ---------------------------------------------------------

final_result = await writer.run(
    task=f"""
Revise your previous blog post using the following aggregated feedback.

Feedback:
{final_feedback}

Requirements:
- Include a suitable title.
- Keep the entire blog post within 100 words.
- Make it concise and engaging.
- Do not include explanations, review notes, or word-count commentary.
- Return only the final blog post.
"""
)

final_blogpost = final_result.messages[-1].content


# ---------------------------------------------------------
# 10. PRINT REVIEWS AND FINAL RESULT
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("GENERAL CRITIC FEEDBACK")
print("=" * 70)
print(critic_feedback)

print("\n" + "=" * 70)
print("SEO REVIEW")
print("=" * 70)
print(seo_feedback)

print("\n" + "=" * 70)
print("LEGAL REVIEW")
print("=" * 70)
print(legal_feedback)

print("\n" + "=" * 70)
print("ETHICS REVIEW")
print("=" * 70)
print(ethics_feedback)

print("\n" + "=" * 70)
print("AGGREGATED FEEDBACK")
print("=" * 70)
print(final_feedback)

print("\n" + "=" * 70)
print("FINAL REFINED BLOG POST")
print("=" * 70)
print(final_blogpost)

print("\nWord count:", len(final_blogpost.split()))

FIRST DRAFT
DeepLearning.AI: Democratizing Practical AI

DeepLearning.AI delivers accessible, project-driven AI education led by Andrew Ng. Its clear courses, hands-on labs and specialization tracks turn beginners into practitioners quickly. Emphasizing ethics, real-world applications and community support, it bridges academia and industry—helping learners build portfolios, land jobs and contribute responsibly to AI. For anyone serious about machine learning, it’s a practical, high-quality launchpad.

GENERAL CRITIC FEEDBACK
You met the brief: title present, concise, under 100 words, and focused on practical education. The piece reads authoritative and highlights ethics and outcomes.

Suggestions to improve clarity and engagement
- Hook: Start with a stronger, more vivid opening to grab readers (e.g., emphasize career impact or a flagship course).
- Specifics: Add a concrete example (course name, project type, or a measurable outcome) to make benefits believable.
- Tone: Vary sentence 